# Stage 1 — Understand the employee attrition data

**Product Owner question:** Do we understand the business label and trust the source enough to begin feature design?

This notebook downloads a public synthetic IBM HR dataset, validates its structure, and explains what must happen before modeling. It is educational and must not be used for real employment decisions.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.stage1_data_profile import load_data, validate_data
df = load_data()
df.head()

## What the code did

It retrieved the CSV from the documented public source and loaded it into a pandas DataFrame—an in-memory table. In a Darwin-style architecture, this is conceptually where curated BigQuery data would enter the ML preparation workflow.

In [ ]:
report = validate_data(df)
report

## Product Owner interpretation

The validation checks row and column counts, label values, employee-key uniqueness, nonnegative business measures, missing values, duplicates, and constant columns. Passing these checks does not prove the data is appropriate for production; it only establishes basic structural readiness.

In [ ]:
attrition_summary = (
    df['Attrition'].value_counts()
      .rename_axis('attrition')
      .to_frame('employees')
)
attrition_summary['percentage'] = (attrition_summary['employees'] / len(df) * 100).round(2)
attrition_summary

## Why class balance matters

Most records are `No`. A model that predicts `No` for everyone could look accurate while failing to identify attrition. Later stages will therefore evaluate precision, recall, F1, ROC-AUC, PR-AUC, and calibration rather than accuracy alone.

In [ ]:
column_profile = df.agg(['count', 'nunique']).T
column_profile['missing'] = df.isna().sum()
column_profile.sort_values(['nunique', 'missing']).head(15)

## Stage 1 decision

The dataset is suitable for an educational prototype. Before Stage 2, we will classify fields as identifiers, constant technical fields, potential features, sensitive review-only fields, and label-only fields.